
> This notebook contains the scripts that were used to pick the stratified samples (from compilation, runtime and test mismatch bugs) for each tool which we manually analysed.




Analyzed COBOL files and extracts COBOL constructs from each program file in our dataset.

In [ ]:
"""
COBOL Construct Extractor
Analyzes COBOL files and extracts all language constructs used
"""

import os
import re
import json
from pathlib import Path
from collections import defaultdict

class COBOLAnalyzer:
    """Extract COBOL constructs from source code"""

    def __init__(self, filepath):
        self.filepath = filepath
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            self.content = f.read().upper()  # COBOL is case-insensitive
        self.lines = self.content.split('\n')

    def extract_constructs(self):
        """Extract all COBOL constructs from the file"""
        constructs = {
            # Data types
            'data_types': self._find_data_types(),

            # Control flow
            'control_flow': self._find_control_flow(),

            # String operations
            'string_operations': self._find_string_operations(),

            # Math operations
            'math_operations': self._find_math_operations(),

            # I/O operations
            'io_operations': self._find_io_operations(),

            # Data structures
            'data_structures': self._find_data_structures(),

            # Advanced features
            'advanced_features': self._find_advanced_features(),

            # File stats
            'file_stats': self._get_file_stats()
        }

        return constructs

    def _find_data_types(self):
        """Find data type declarations"""
        types = {
            'PIC_X': len(re.findall(r'\bPIC\s+X', self.content)),
            'PIC_9': len(re.findall(r'\bPIC\s+9', self.content)),
            'PIC_S9': len(re.findall(r'\bPIC\s+S9', self.content)),
            'PIC_V9': len(re.findall(r'\bPIC\s+.*V.*9', self.content)),
            'COMP': len(re.findall(r'\bCOMP\b', self.content)),
            'COMP-3': len(re.findall(r'\bCOMP-3\b', self.content)),
            'BINARY': len(re.findall(r'\bBINARY\b', self.content)),
            'PACKED-DECIMAL': len(re.findall(r'\bPACKED-DECIMAL\b', self.content)),
        }
        return {k: v for k, v in types.items() if v > 0}

    def _find_control_flow(self):
        """Find control flow constructs"""
        flow = {
            'PERFORM': len(re.findall(r'\bPERFORM\b', self.content)),
            'PERFORM_TIMES': len(re.findall(r'\bPERFORM\s+.*\s+TIMES\b', self.content)),
            'PERFORM_UNTIL': len(re.findall(r'\bPERFORM\s+.*\s+UNTIL\b', self.content)),
            'PERFORM_VARYING': len(re.findall(r'\bPERFORM\s+.*\s+VARYING\b', self.content)),
            'PERFORM_THRU': len(re.findall(r'\bPERFORM\s+.*\s+THRU\b', self.content)),
            'IF': len(re.findall(r'\bIF\b', self.content)),
            'IF_ELSE': len(re.findall(r'\bELSE\b', self.content)),
            'EVALUATE': len(re.findall(r'\bEVALUATE\b', self.content)),
            'GO_TO': len(re.findall(r'\bGO\s+TO\b', self.content)),
            'EXIT': len(re.findall(r'\bEXIT\b', self.content)),
            'STOP_RUN': len(re.findall(r'\bSTOP\s+RUN\b', self.content)),
        }
        return {k: v for k, v in flow.items() if v > 0}

    def _find_string_operations(self):
        """Find string manipulation operations"""
        ops = {
            'STRING': len(re.findall(r'\bSTRING\b', self.content)),
            'UNSTRING': len(re.findall(r'\bUNSTRING\b', self.content)),
            'INSPECT': len(re.findall(r'\bINSPECT\b', self.content)),
            'INSPECT_REPLACING': len(re.findall(r'\bINSPECT\s+.*\s+REPLACING\b', self.content)),
            'INSPECT_TALLYING': len(re.findall(r'\bINSPECT\s+.*\s+TALLYING\b', self.content)),
            'MOVE': len(re.findall(r'\bMOVE\b', self.content)),
            'INITIALIZE': len(re.findall(r'\bINITIALIZE\b', self.content)),
        }
        return {k: v for k, v in ops.items() if v > 0}

    def _find_math_operations(self):
        """Find mathematical operations"""
        ops = {
            'ADD': len(re.findall(r'\bADD\b', self.content)),
            'SUBTRACT': len(re.findall(r'\bSUBTRACT\b', self.content)),
            'MULTIPLY': len(re.findall(r'\bMULTIPLY\b', self.content)),
            'DIVIDE': len(re.findall(r'\bDIVIDE\b', self.content)),
            'COMPUTE': len(re.findall(r'\bCOMPUTE\b', self.content)),
            'REMAINDER': len(re.findall(r'\bREMAINDER\b', self.content)),
            'ROUNDED': len(re.findall(r'\bROUNDED\b', self.content)),
            'ON_SIZE_ERROR': len(re.findall(r'\bON\s+SIZE\s+ERROR\b', self.content)),
        }
        return {k: v for k, v in ops.items() if v > 0}

    def _find_io_operations(self):
        """Find I/O operations"""
        ops = {
            'ACCEPT': len(re.findall(r'\bACCEPT\b', self.content)),
            'DISPLAY': len(re.findall(r'\bDISPLAY\b', self.content)),
            'READ': len(re.findall(r'\bREAD\b', self.content)),
            'WRITE': len(re.findall(r'\bWRITE\b', self.content)),
            'OPEN': len(re.findall(r'\bOPEN\b', self.content)),
            'CLOSE': len(re.findall(r'\bCLOSE\b', self.content)),
        }
        return {k: v for k, v in ops.items() if v > 0}

    def _find_data_structures(self):
        """Find data structure constructs"""
        structs = {
            'OCCURS': len(re.findall(r'\bOCCURS\b', self.content)),
            'OCCURS_DEPENDING': len(re.findall(r'\bOCCURS\s+.*\s+DEPENDING\b', self.content)),
            'REDEFINES': len(re.findall(r'\bREDEFINES\b', self.content)),
            'LEVEL_88': len(re.findall(r'\b88\s+', self.content)),
            'LEVEL_77': len(re.findall(r'\b77\s+', self.content)),
            'LEVEL_01': len(re.findall(r'\b01\s+', self.content)),
            'VALUE': len(re.findall(r'\bVALUE\b', self.content)),
        }
        return {k: v for k, v in structs.items() if v > 0}

    def _find_advanced_features(self):
        """Find advanced COBOL features"""
        features = {
            'FUNCTION': len(re.findall(r'\bFUNCTION\b', self.content)),
            'FUNCTION_INTEGER': len(re.findall(r'\bFUNCTION\s+INTEGER\b', self.content)),
            'FUNCTION_MOD': len(re.findall(r'\bFUNCTION\s+MOD\b', self.content)),
            'FUNCTION_LENGTH': len(re.findall(r'\bFUNCTION\s+LENGTH\b', self.content)),
            'FUNCTION_MAX': len(re.findall(r'\bFUNCTION\s+MAX\b', self.content)),
            'FUNCTION_MIN': len(re.findall(r'\bFUNCTION\s+MIN\b', self.content)),
            'CALL': len(re.findall(r'\bCALL\b', self.content)),
            'POINTER': len(re.findall(r'\bPOINTER\b', self.content)),
            'REFERENCE': len(re.findall(r'\bREFERENCE\b', self.content)),
        }
        return {k: v for k, v in features.items() if v > 0}

    def _get_file_stats(self):
        """Get basic file statistics"""
        code_lines = [l for l in self.lines if l.strip() and not l.strip().startswith('*')]

        return {
            'total_lines': len(self.lines),
            'code_lines': len(code_lines),
            'has_identification_division': 'IDENTIFICATION DIVISION' in self.content,
            'has_data_division': 'DATA DIVISION' in self.content,
            'has_procedure_division': 'PROCEDURE DIVISION' in self.content,
        }

def analyze_all_cobol_files(base_dir, output_file='/content/cobol_constructs.json'):
    """
    Analyze all COBOL files in the codenet_final directory structure
    """
    base_path = Path(base_dir)

    if not base_path.exists():
        print(f" Directory not found: {base_dir}")
        return None

    print(f"\n{'='*80}")
    print(f"Analyzing COBOL files in: {base_dir}")
    print(f"{'='*80}\n")

    results = {}
    stats = {
        'total_problems': 0,
        'successful': 0,
        'failed': 0,
        'no_cobol_file': 0
    }

    # Iterate through each PID folder
    for pid_folder in sorted(base_path.iterdir()):
        if not pid_folder.is_dir():
            continue

        pid = pid_folder.name
        stats['total_problems'] += 1

        # Look for cobol subfolder
        cobol_folder = pid_folder / 'COBOL'

        if not cobol_folder.exists() or not cobol_folder.is_dir():
            print(f"  {pid}: No cobol folder found")
            stats['no_cobol_file'] += 1
            continue

        # Get the first .cob file
        cob_files = sorted(list(cobol_folder.glob("*.cob")))

        if not cob_files:
            print(f"  {pid}: No .cob file found in cobol folder")
            stats['no_cobol_file'] += 1
            continue

        # Analyze the first .cob file
        cob_file = cob_files[0]

        try:
            analyzer = COBOLAnalyzer(cob_file)
            constructs = analyzer.extract_constructs()

            results[pid] = {
                'file_path': str(cob_file.relative_to(base_path)),
                'file_name': cob_file.name,
                'constructs': constructs
            }

            stats['successful'] += 1

            if stats['successful'] % 50 == 0:
                print(f" Analyzed {stats['successful']} files...")

        except Exception as e:
            print(f" {pid}: Error analyzing {cob_file.name}: {str(e)}")
            stats['failed'] += 1

    # Save results to JSON
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)

    # Print summary
    print(f"\n{'='*80}")
    print(f"ANALYSIS COMPLETE")
    print(f"{'='*80}")
    print(f"Total problems: {stats['total_problems']}")
    print(f"Successfully analyzed: {stats['successful']}")
    print(f"Failed to analyze: {stats['failed']}")
    print(f"No COBOL file found: {stats['no_cobol_file']}")
    print(f"\nResults saved to: {output_file}")
    print(f"{'='*80}\n")

    # Print construct summary
    print_construct_summary(results)

    return results

def print_construct_summary(results):
    """Print a summary of all constructs found"""
    print(f"\n{'='*80}")
    print(f"COBOL CONSTRUCT SUMMARY")
    print(f"{'='*80}\n")

    # Aggregate construct counts
    all_constructs = defaultdict(int)

    for pid, data in results.items():
        constructs = data['constructs']
        for category, items in constructs.items():
            if isinstance(items, dict):
                for construct, count in items.items():
                    all_constructs[f"{category}.{construct}"] += 1  # Count files using this construct

    # Sort by frequency
    sorted_constructs = sorted(all_constructs.items(), key=lambda x: x[1], reverse=True)

    print(f"{'Construct':<40} {'Files Using It':>20}")
    print(f"{'-'*40} {'-'*20}")

    for construct, count in sorted_constructs[:30]:  # Top 30
        print(f"{construct:<40} {count:>20}")

    print(f"\n(Showing top 30 most common constructs)")

if __name__ == "__main__":
    # Mount Google Drive (for Colab)

    base_dir = "CodeNet_Dataset"
    output_file = "OUTPUT DIR / cobol_constructs.json"

    # Analyze all COBOL files
    results = analyze_all_cobol_files(base_dir, output_file)

Takes the tool_functional_test_result.json, and extracts compilation, runtime, test mismatch errors along with their error message, and saves to errors_tool.json.

In [ ]:
"""
Extract Error Messages from Translation Results
Creates a simple mapping of problem_id -> error_message for failed translations
"""

import json
from pathlib import Path

def extract_errors(translation_json_path, output_path='errors_extracted.json'):
    """
    Extract all error messages from translation results
    """
    # Load translation results
    with open(translation_json_path, 'r') as f:
        data = json.load(f)

    # Handle different JSON structures
    if isinstance(data, dict):
        # If it's a dict, look for a 'results' key or similar
        if 'detailed_results' in data:
            results = data['detailed_results']
        elif 'results' in data:
            results = data['results']
        elif 'test_results' in data:
            results = data['test_results']
        else:
            # Assume the dict itself is the results (pid as keys)
            print("  JSON structure unclear. Showing first few keys:")
            print(list(data.keys())[:10])
            raise ValueError("Cannot find results in JSON. Please check structure.")
    else:
        results = data

    errors = {}

    stats = {
        'total': 0,
        'passed': 0,
        'compilation_errors': 0,
        'runtime_errors': 0,
        'test_errors': 0
    }

    # Process each result
    for result in results:
        pid = result['problem_id']
        stats['total'] += 1

        # Check if it passed
        if result['overall_status'] == 'passed':
            stats['passed'] += 1
            continue

        # Collect error information
        error_info = {
            'problem_id': pid,
            'java_file': result.get('java_file'),
            'error_type': None,
            'error_message': None
        }

        # Check compilation error
        if result['compilation']['success'] == False:
            error_info['error_type'] = 'compilation'
            error_info['error_message'] = result['compilation']['error']
            stats['compilation_errors'] += 1

        # Check runtime error (only if compilation succeeded)
        elif result['execution']['success'] == False:
            error_info['error_type'] = 'runtime'
            error_info['error_message'] = result['execution'].get('error', 'Runtime error (no details)')

            # Add execution time if available (for timeouts)
            if result['execution'].get('time_seconds') is not None:
                error_info['execution_time'] = result['execution']['time_seconds']

            stats['runtime_errors'] += 1

        # Check test validation error (only if execution succeeded)
        elif result['test_validation']['success'] == False:
            error_info['error_type'] = 'test_validation'
            error_info['error_message'] = result['test_validation'].get('error', 'Test validation failed (no details)')

            # Include actual output if available
            if result['execution'].get('output') is not None:
                error_info['actual_output'] = result['execution']['output']

            stats['test_errors'] += 1

        # Store the error
        errors[pid] = error_info

    # Save to JSON
    with open(output_path, 'w') as f:
        json.dump(errors, f, indent=2)

    # Print summary
    print(f"\n{'='*80}")
    print(f"ERROR EXTRACTION SUMMARY")
    print(f"{'='*80}")
    print(f"Total problems: {stats['total']}")
    print(f"Passed: {stats['passed']} ({stats['passed']/stats['total']*100:.1f}%)")
    print(f"\nErrors extracted: {len(errors)}")
    print(f"  - Compilation errors: {stats['compilation_errors']} ({stats['compilation_errors']/stats['total']*100:.1f}%)")
    print(f"  - Runtime errors: {stats['runtime_errors']} ({stats['runtime_errors']/stats['total']*100:.1f}%)")
    print(f"  - Test validation errors: {stats['test_errors']} ({stats['test_errors']/stats['total']*100:.1f}%)")
    print(f"\nSaved to: {output_path}")
    print(f"{'='*80}\n")

    # Show some examples
    print("EXAMPLE ERRORS:\n")

    # Show one of each type
    for error_type in ['compilation', 'runtime', 'test_validation']:
        example = next((e for e in errors.values() if e['error_type'] == error_type), None)
        if example:
            print(f"{error_type.upper()} ERROR EXAMPLE:")
            print(f"  Problem ID: {example['problem_id']}")
            print(f"  Java file: {example['java_file']}")
            print(f"  Error message (first 200 chars):")
            print(f"    {example['error_message'][:200]}...")
            print()

    return errors, stats

if __name__ == "__main__":
  
    input_file = "Dissertation/CodeNet_Dataset/ TOOL NAME_functional_test_result.json" #replace with actual path
    output_file = "OUTPUT DIR /errors_ TOOL NAME .json" #replace with actual path

    # Check if file exists
    if not Path(input_file).exists():
        print(f" Input file not found: {input_file}")
        import sys
        sys.exit(1)

    # Extract errors
    errors, stats = extract_errors(input_file, output_file)

    print(f"Extracted {len(errors)} error messages")
    print(f"Saved to {output_file}")

Stratified sample builder

In [ ]:
"""
Bug Taxonomy Builder with Smart Sampling
Analyzes translation failures and generates sampling recommendations
"""

import json
import re
from pathlib import Path
from collections import defaultdict

class BugTaxonomyAnalyzer:
    """Analyze failures and group by patterns and COBOL constructs"""

    def __init__(self, errors_file, constructs_file, tool_name):
        self.tool_name = tool_name

        # Load errors
        with open(errors_file, 'r') as f:
            self.errors = json.load(f)

        # Load COBOL constructs
        with open(constructs_file, 'r') as f:
            self.constructs = json.load(f)

    def analyze(self):
        """Main analysis function"""
        print(f"\n{'='*80}")
        print(f"Analyzing {self.tool_name} Failures")
        print(f"{'='*80}\n")

        # Categorize errors
        compilation_errors = {}
        runtime_errors = {}
        test_failures = {}

        for pid, error_data in self.errors.items():
            error_type = error_data['error_type']

            if error_type == 'compilation':
                self._process_compilation_error(pid, error_data, compilation_errors)
            elif error_type == 'runtime':
                self._process_runtime_error(pid, error_data, runtime_errors)
            elif error_type == 'test_validation':
                self._process_test_failure(pid, error_data, test_failures)

        # Build taxonomy
        taxonomy = {
            'tool_name': self.tool_name,
            'total_failures': len(self.errors),
            'compilation_errors': self._group_compilation_errors(compilation_errors),
            'runtime_errors': self._group_runtime_errors(runtime_errors),
            'test_failures': self._group_test_failures(test_failures)
        }

        return taxonomy

    def _process_compilation_error(self, pid, error_data, storage):
        """Process compilation error and extract pattern"""
        error_msg = error_data['error_message']

        # Extract error pattern
        patterns = {
            'type_mismatch': r'incompatible types|lossy conversion|cannot convert',
            'undefined_symbol': r'cannot find symbol|cannot resolve',
            'syntax_error': r'expected|illegal start|not a statement',
            'access_error': r'has private access|is not visible',
            'duplicate_error': r'already defined|duplicate',
            'import_error': r'package .* does not exist',
        }

        matched_pattern = 'other'
        for pattern_name, regex in patterns.items():
            if re.search(regex, error_msg, re.IGNORECASE):
                matched_pattern = pattern_name
                break

        if matched_pattern not in storage:
            storage[matched_pattern] = []

        storage[matched_pattern].append({
            'pid': pid,
            'error_msg': error_msg,
            'constructs': self._get_constructs(pid)
        })

    def _process_runtime_error(self, pid, error_data, storage):
        """Process runtime error"""
        error_msg = error_data['error_message']

        # Determine if timeout or exception
        if 'timeout' in error_msg.lower():
            error_category = 'timeout'
        elif 'ArrayIndexOutOfBoundsException' in error_msg:
            error_category = 'array_index_exception'
        elif 'NullPointerException' in error_msg:
            error_category = 'null_pointer'
        elif 'NumberFormatException' in error_msg:
            error_category = 'number_format'
        elif 'ArithmeticException' in error_msg:
            error_category = 'arithmetic_exception'
        else:
            error_category = 'other_exception'

        if error_category not in storage:
            storage[error_category] = []

        storage[error_category].append({
            'pid': pid,
            'error_msg': error_msg,
            'constructs': self._get_constructs(pid)
        })

    def _process_test_failure(self, pid, error_data, storage):
        """Process test validation failure"""
        # Group by constructs since we don't have line-level info
        constructs = self._get_constructs(pid)

        # Create a key based on major constructs
        construct_key = self._create_construct_key(constructs)

        if construct_key not in storage:
            storage[construct_key] = []

        storage[construct_key].append({
            'pid': pid,
            'error_msg': error_data.get('error_message', 'Output mismatch'),
            'constructs': constructs
        })

    def _get_constructs(self, pid):
        """Get COBOL constructs for a problem"""
        if pid not in self.constructs:
            return {}

        constructs_data = self.constructs[pid]['constructs']

        # Flatten constructs into a simple dict
        flattened = {}
        for category, items in constructs_data.items():
            if isinstance(items, dict):
                for construct, count in items.items():
                    if count > 0:
                        flattened[f"{category}.{construct}"] = count

        return flattened

    def _create_construct_key(self, constructs):
        """Create a key based on significant constructs"""
        # Priority constructs that likely cause issues
        priority_constructs = [
            'string_operations.UNSTRING',
            'string_operations.STRING',
            'string_operations.INSPECT',
            'data_structures.OCCURS',
            'data_structures.REDEFINES',
            'control_flow.PERFORM_VARYING',
            'control_flow.PERFORM_UNTIL',
            'math_operations.COMPUTE',
            'advanced_features.FUNCTION',
            'data_types.COMP',
        ]

        present = [c for c in priority_constructs if c in constructs]

        if not present:
            return 'basic'

        # Use top 2 constructs as key
        return '_'.join(present[:2])

    def _group_compilation_errors(self, compilation_errors):
        """Group and analyze compilation errors"""
        grouped = {}

        for pattern, errors in compilation_errors.items():
            # Find common constructs
            construct_counts = defaultdict(int)
            for error in errors:
                for construct in error['constructs'].keys():
                    construct_counts[construct] += 1

            # Get most common constructs (appear in >50% of errors)
            threshold = len(errors) * 0.5
            common_constructs = [c for c, count in construct_counts.items() if count >= threshold]

            # Select diverse samples
            samples = self._select_diverse_samples(errors, num_samples=5)

            grouped[pattern] = {
                'count': len(errors),
                'common_constructs': sorted(common_constructs, key=lambda x: construct_counts[x], reverse=True)[:5],
                'sample_pids': [s['pid'] for s in samples],
                'sample_details': samples
            }

        return grouped

    def _group_runtime_errors(self, runtime_errors):
        """Group and analyze runtime errors"""
        grouped = {}

        for category, errors in runtime_errors.items():
            # Find common constructs
            construct_counts = defaultdict(int)
            for error in errors:
                for construct in error['constructs'].keys():
                    construct_counts[construct] += 1

            threshold = len(errors) * 0.5
            common_constructs = [c for c, count in construct_counts.items() if count >= threshold]

            samples = self._select_diverse_samples(errors, num_samples=5)

            grouped[category] = {
                'count': len(errors),
                'common_constructs': sorted(common_constructs, key=lambda x: construct_counts[x], reverse=True)[:5],
                'sample_pids': [s['pid'] for s in samples],
                'sample_details': samples
            }

        return grouped

    def _group_test_failures(self, test_failures):
        """Group and analyze test failures by construct patterns"""
        grouped = {}

        for construct_key, errors in test_failures.items():
            # Find common constructs within this group
            construct_counts = defaultdict(int)
            for error in errors:
                for construct in error['constructs'].keys():
                    construct_counts[construct] += 1

            threshold = len(errors) * 0.5
            common_constructs = [c for c, count in construct_counts.items() if count >= threshold]

            samples = self._select_diverse_samples(errors, num_samples=5)

            grouped[construct_key] = {
                'count': len(errors),
                'common_constructs': sorted(common_constructs, key=lambda x: construct_counts[x], reverse=True)[:5],
                'sample_pids': [s['pid'] for s in samples],
                'sample_details': samples
            }

        return grouped

    def _select_diverse_samples(self, errors, num_samples=5):
        """Select diverse sample PIDs that represent different complexity levels"""
        if len(errors) <= num_samples:
            return errors

        # Sort by construct complexity (number of different constructs)
        sorted_errors = sorted(errors, key=lambda x: len(x['constructs']))

        # Select from different complexity levels
        samples = []
        step = len(sorted_errors) // num_samples

        for i in range(num_samples):
            idx = min(i * step, len(sorted_errors) - 1)
            samples.append(sorted_errors[idx])

        return samples


def generate_sampling_report(taxonomy, output_file):
    """Generate human-readable sampling recommendations"""
    tool_name = taxonomy['tool_name']
    total = taxonomy['total_failures']

    report = []
    report.append("=" * 80)
    report.append(f"{tool_name.upper()} BUG TAXONOMY - SAMPLING RECOMMENDATIONS")
    report.append("=" * 80)
    report.append(f"\nTotal Failures: {total}")

    # Calculate recommended sample size
    total_samples = 0

    # Compilation errors
    comp_errors = taxonomy['compilation_errors']
    if comp_errors:
        comp_total = sum(v['count'] for v in comp_errors.values())
        report.append(f"\n{'-'*80}")
        report.append(f"PRIORITY 1: COMPILATION ERRORS ({comp_total} total)")
        report.append(f"{'-'*80}\n")

        for group_name, data in comp_errors.items():
            report.append(f"Group: {group_name.replace('_', ' ').title()} ({data['count']} files)")
            report.append(f"  Common COBOL constructs: {', '.join(data['common_constructs'][:3])}")
            report.append(f"\n  INSPECT THESE {len(data['sample_pids'])} FILES:")

            for i, sample in enumerate(data['sample_details'], 1):
                constructs_str = ', '.join(list(sample['constructs'].keys())[:3])
                report.append(f"    {i}. {sample['pid']} - Has: {constructs_str}")
                total_samples += 1

            report.append("")

    # Runtime errors
    runtime_errors = taxonomy['runtime_errors']
    if runtime_errors:
        runtime_total = sum(v['count'] for v in runtime_errors.values())
        report.append(f"\n{'-'*80}")
        report.append(f"PRIORITY 2: RUNTIME ERRORS ({runtime_total} total)")
        report.append(f"{'-'*80}\n")

        for group_name, data in runtime_errors.items():
            report.append(f"Group: {group_name.replace('_', ' ').title()} ({data['count']} files)")
            report.append(f"  Common COBOL constructs: {', '.join(data['common_constructs'][:3])}")
            report.append(f"\n INSPECT THESE {len(data['sample_pids'])} FILES:")

            for i, sample in enumerate(data['sample_details'], 1):
                constructs_str = ', '.join(list(sample['constructs'].keys())[:3])
                report.append(f"    {i}. {sample['pid']} - Has: {constructs_str}")
                total_samples += 1

            report.append("")

    # Test failures
    test_failures = taxonomy['test_failures']
    if test_failures:
        test_total = sum(v['count'] for v in test_failures.values())
        report.append(f"\n{'-'*80}")
        report.append(f"PRIORITY 3: TEST FAILURES - WRONG OUTPUT ({test_total} total)")
        report.append(f"{'-'*80}\n")

        # Show top 5 groups
        sorted_groups = sorted(test_failures.items(), key=lambda x: x[1]['count'], reverse=True)

        for group_name, data in sorted_groups[:5]:
            report.append(f"Group: {group_name.replace('_', ' ')} ({data['count']} files)")
            report.append(f"  Common COBOL constructs: {', '.join(data['common_constructs'][:3])}")
            report.append(f"\n  INSPECT THESE {min(len(data['sample_pids']), 5)} FILES:")

            for i, sample in enumerate(data['sample_details'][:5], 1):
                constructs_str = ', '.join(list(sample['constructs'].keys())[:3])
                report.append(f"    {i}. {sample['pid']} - Has: {constructs_str}")
                total_samples += 1

            report.append("")

    report.append(f"\n{'='*80}")
    report.append(f"TOTAL RECOMMENDED SAMPLES: {total_samples} files")
    report.append(f"Coverage: {total_samples}/{total} failures ({total_samples/total*100:.1f}%)")
    report.append(f"{'='*80}")

    # Save report
    with open(output_file, 'w') as f:
        f.write('\n'.join(report))

    # Print to console
    print('\n'.join(report))


def analyze_all_tools():
    """Analyze all three tools"""


    base_path = "."

    tools = [
        #('qwen', f'{base_path}/ErrorLinesExtracted/errors_qwen.json'),
        ('xmainframe', f'{base_path}/ErrorLinesExtracted/errors_xmainframe.json')
        #('nonllm', f'{base_path}/ErrorLinesExtracted/errors_nonllm.json'),
    ]

    constructs_file = f'{base_path}/ErrorLinesExtracted/cobol_constructs.json'

    # Create output directory in Drive
    output_dir = f'{base_path}/ErrorLinesExtracted/results'
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    print(f"Output directory: {output_dir}\n")

    # Check if constructs file exists
    if not Path(constructs_file).exists():
        print(f"❌ COBOL constructs file not found: {constructs_file}")
        print("Please run extract_cobol_constructs.py first!")
        return

    all_taxonomies = {}

    for tool_name, errors_file in tools:
        if not Path(errors_file).exists():
            print(f" Skipping {tool_name}: {errors_file} not found")
            continue

        print(f"\n{'='*80}")
        print(f"Processing {tool_name}...")
        print(f"{'='*80}")

        # Analyze
        analyzer = BugTaxonomyAnalyzer(errors_file, constructs_file, tool_name)
        taxonomy = analyzer.analyze()
        all_taxonomies[tool_name] = taxonomy

        # Save detailed taxonomy JSON
        json_output = f'{output_dir}/taxonomy_{tool_name}.json'
        with open(json_output, 'w') as f:
            json.dump(taxonomy, f, indent=2)
        print(f"\n✓ Saved detailed taxonomy to: {json_output}")

        # Generate sampling report
        report_output = f'{output_dir}/sampling_recommendations_{tool_name}.txt'
        generate_sampling_report(taxonomy, report_output)
        print(f"✓ Saved sampling report to: {report_output}")

    # Generate comparison summary
    if len(all_taxonomies) > 1:
        comparison_file = f'{output_dir}/tool_comparison.txt'
        generate_comparison_report(all_taxonomies, comparison_file)
        print(f"\n✓ Saved tool comparison to: {comparison_file}")

    print(f"\n{'='*80}")
    print("Analysis Complete!")
    print(f"All results saved to: {output_dir}")
    print(f"{'='*80}")


def generate_comparison_report(all_taxonomies, output_file):
    """Generate comparison report across all tools"""
    report = []
    report.append("=" * 80)
    report.append("CROSS-TOOL FAILURE COMPARISON")
    report.append("=" * 80)

    # Summary table
    report.append("\n" + "-" * 80)
    report.append(f"{'Tool':<15} {'Total Fails':<15} {'Compilation':<15} {'Runtime':<15} {'Test':<15}")
    report.append("-" * 80)

    for tool_name, taxonomy in all_taxonomies.items():
        comp_count = sum(v['count'] for v in taxonomy['compilation_errors'].values())
        runtime_count = sum(v['count'] for v in taxonomy['runtime_errors'].values())
        test_count = sum(v['count'] for v in taxonomy['test_failures'].values())

        report.append(f"{tool_name:<15} {taxonomy['total_failures']:<15} {comp_count:<15} {runtime_count:<15} {test_count:<15}")

    report.append("-" * 80)

    # Save
    with open(output_file, 'w') as f:
        f.write('\n'.join(report))

    print('\n'.join(report))


if __name__ == "__main__":
    analyze_all_tools()

Print the original COBOL file, translated Java, and error messages for a given PID and Translator - to facilitate the manual examination of bugs.

In [ ]:
import os
from pathlib import Path

def read_cobol_and_java_files(pid):
    print("\n" + "=" * 80)
    print(f"PID: {pid}")
    print("=" * 80)

    # ===== READ COBOL FILE =====
    print("\n" + "-" * 80)
    print("COBOL FILE")
    print("-" * 80)

    cobol_base_path = Path("CodeNet_Dataset")
    cobol_path = cobol_base_path / pid / "COBOL"

    if not cobol_path.exists():
        print(f"Error: Path does not exist: {cobol_path}")
    else:
        cob_files = sorted(list(cobol_path.glob("*.cob")))

        if not cob_files:
            print(f"Error: No .cob files found in {cobol_path}")
        else:
            first_cob = cob_files[0]
            print(f"Reading file: {first_cob}")
            print()

            with open(first_cob, 'r') as f:
                cobol_content = f.read()
                print(cobol_content)

            print()
            print(f"File: {first_cob.name}")
            print(f"Total lines: {len(cobol_content.splitlines())}")
            print(f"Total characters: {len(cobol_content)}")

    # ===== READ JAVA FILE =====
    print("\n" + "-" * 80)
    print("JAVA FILE (TRANSLATED)")
    print("-" * 80)

    java_base_path = Path("PATH TO TRANSLATION RESULTS OF A TOOL")
    java_path = java_base_path / pid

    if not java_path.exists():
        print(f"Error: Path does not exist: {java_path}")
    else:
        java_files = sorted(list(java_path.glob("*.java")))

        if not java_files:
            print(f"Error: No .java files found in {java_path}")
        else:
            first_java = java_files[0]
            print(f"Reading file: {first_java}")
            print()

            with open(first_java, 'r') as f:
                java_content = f.read()
                print(java_content)

            print()
            print(f"File: {first_java.name}")
            print(f"Total lines: {len(java_content.splitlines())}")
            print(f"Total characters: {len(java_content)}")

    # ===== READ ERROR FROM JSON =====
    print("\n" + "-" * 80)
    print("COMPILATION / RUNTIME ERROR")
    print("-" * 80)

    error_json_path = Path("PATH TO ERROR MESSAGE - /errors_COBOL4J.json")

    if not error_json_path.exists():
        print(f"Error: File does not exist: {error_json_path}")
    else:
        import json
        with open(error_json_path, 'r') as f:
            error_data = json.load(f)

        if pid not in error_data:
            print(f"No entry found for {pid}")
        else:
            entry = error_data[pid]
            print(f"Error Type    : {entry.get('error_type', 'N/A')}")
            print(f"Error Details : {entry.get('error_message', entry.get('error_details', 'N/A'))}")
            # Print any other fields in the entry
            for k, v in entry.items():
                if k not in ('error_type', 'error_message', 'error_details'):
                    print(f"{k:14}: {v}")


def read_multiple_pids(pid_list):
    print("=" * 80)
    print(f"PROCESSING {len(pid_list)} PIDs")
    print("=" * 80)

    for i, pid in enumerate(pid_list, 1):
        print(f"\n\n{'#' * 80}")
        print(f"# Processing PID {i}/{len(pid_list)}")
        print(f"{'#' * 80}")
        read_cobol_and_java_files(pid)

    print("\n\n" + "=" * 80)
    print("COMPLETED ALL PIDs")
    print("=" * 80)


if __name__ == "__main__":
    pids = ["p03556"]  # Add as many PIDs as you want
    read_multiple_pids(pids)